In [1]:
import json
import os 

with open('config.json', 'r') as f:
    data = json.load(f)

pathway_gen = os.path.abspath(data["python_files"])
pathway_temp = os.path.abspath(data["publications"])
pathway = os.path.join(pathway_temp, "barth2006tracking")
original_data_pathway = os.path.join(pathway, "original_data")

complete_path_1 = os.path.join(original_data_pathway, "barth_001_Best.csv")

out_pathway = os.path.join(pathway, "standardized_data")
if not os.path.exists(out_pathway):
    os.makedirs(out_pathway)

In [2]:
import pandas as pd
import numpy as np
import pyreadstat

df = pd.read_csv(complete_path_1)

df['study_id']="barth2006tracking"
df.columns = map(str.lower, df.columns)
df=df.applymap(lambda s: s.lower() if type(s) == str else s)


In [3]:
df.rename(columns={"subject": "participant",
                   "species":"species_original",
                   "sex":"sex_original",
                   "age":"age_original",
                   "delres":"delay_task",
                   "inhib":"inhibition_task",
                   "anotb":"a_not_b_task",
                   "rotate":"rotation_task",
                   "trans":"transposition_task",
                   "op":"object_permanence_task",
                   "total":"total_tasks"}, inplace=True)

In [4]:
comp_path_name_errors = os.path.join(pathway_gen, "common_name_errors.csv")

df_name  = pd.read_csv(comp_path_name_errors)
df['participant'] = df['participant'].str.rstrip()
for x,y in zip(df_name['wrong'],df_name['right']):
    df['participant'].replace(x, y, inplace=True)

comp_path_ape_info = os.path.join(pathway_gen, "apes_includeindatabase.csv")
apedf = pd.read_csv(comp_path_ape_info)    
df= df.merge(apedf,left_on='participant', right_on='name', how='left')


complete_path_age = os.path.join(original_data_pathway, "subject_list.csv")
subject_list = pd.read_csv(complete_path_age)   
df= df.merge(subject_list,left_on='participant', right_on='name', how='left')
df.rename(columns={"age": "age_in_years"}, inplace=True)

df.loc[df.participant == 'dorien', ['drop_out']] = 'true'

# df.columns

In [5]:
studyID_standardized=df[['study_id','participant', 'age_in_years', 'sex', 'species',
       'weight',  'delay_task', 'inhibition_task',
       'a_not_b_task', 'rotation_task', 'transposition_task',
       'object_permanence_task', 'total_tasks', 'drop_out']]
comp_out_path_stand = os.path.join(out_pathway, 'barth2006tracking_exp1_standardized.csv')
studyID_standardized.to_csv(comp_out_path_stand, encoding='utf-8-sig', index=False)


names =studyID_standardized.columns.tolist()
df = pd.DataFrame(names)
df = df.rename(columns={0: "column_name"})
df["description"] = ""
studyID_glossary=df[["column_name", "description"]]

comp_out_path_glossary = os.path.join(out_pathway, 'barth2006tracking_exp1_glossary.csv')
studyID_glossary.to_csv(comp_out_path_glossary, encoding='utf-8-sig', index=False)
